# Notebook 01 - Warehouse Full-History EDA

Nguon: `EDA_CURATED_PLAN.md`, `discuss/eda-curated-implementation/` (file 01-12). Wave A: mo ta toan bo
lich su da hop nhat trong warehouse - KHONG phai causal training dataset.

**PHAI chay qua `python run_wave_a.py`** (tu thu muc `eda/`), KHONG mo notebook nay truc tiep trong
Jupyter roi bam Run All - runner moi la noi tao `analysis_dir` (fail-if-exists) va dat cac bien moi
truong `EDA_SRC_DIR`/`EDA_ANALYSIS_DIR` ma cell duoi day can. Xem `notebooks/README.md`.

Moi truy van DB deu qua `queries.run_metric()` (co kiem `output_schema`), moi ket noi la READ-ONLY
(server-enforced), va MOI bang gia/availability/reference/turnover la ket qua `GROUP BY` thang trong SQL
(bounded-memory: khong frame observation-level nao trong Python). Notebook nay chi orchestration/ve hinh -
logic that nam trong `src/wave_a.py`, `src/queries.py`, `src/metrics.py`, `src/plots.py`, `src/report.py`.

In [ ]:
import os
import sys
from pathlib import Path

# GPT review 12 eda B1: KHONG doan Path.cwd() - doc tu bien moi truong runner da dat truoc khi mo
# kernel. Fail RO RANG neu thieu, khong fallback CWD (nguon loi cu).
try:
    SRC_DIR = Path(os.environ["EDA_SRC_DIR"])
    ANALYSIS_DIR = Path(os.environ["EDA_ANALYSIS_DIR"])
except KeyError as exc:
    raise RuntimeError(
        "Thieu bien moi truong "
        + str(exc)
        + " - notebook nay PHAI duoc chay qua `python run_wave_a.py` (tu thu muc eda/), "
        "khong mo truc tiep trong Jupyter. Xem notebooks/README.md."
    ) from exc

# 7 bien nay la TUY CHON - runner chi dat khi caller (vd test tren fixture disposable) truyen override; neu khong
# co thi cac ham `wave_a.*` ben duoi tu dung default cua chinh no (snapshot warehouse hien hanh).
def _env_path(name: str) -> Path | None:
    value = os.environ.get(name)
    return Path(value) if value else None

EDA_POINTER_PATH = _env_path("EDA_POINTER_PATH")
EDA_OWNERSHIP_MANIFEST_PATH = _env_path("EDA_OWNERSHIP_MANIFEST_PATH")
EDA_COHORT_HISTORY_PATH = _env_path("EDA_COHORT_HISTORY_PATH")
EDA_COHORT_HISTORY_BASE_DIR = _env_path("EDA_COHORT_HISTORY_BASE_DIR")
EDA_VN_HOLIDAYS_CSV = _env_path("EDA_VN_HOLIDAYS_CSV")
EDA_WAREHOUSE_VALIDATION_REPORT_PATH = _env_path("EDA_WAREHOUSE_VALIDATION_REPORT_PATH")
EDA_SOURCE_MANIFEST_PATH = _env_path("EDA_SOURCE_MANIFEST_PATH")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import artifacts
import metrics
import plots
import publication
import queries
import wave_a

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

FIGURES_DIR = ANALYSIS_DIR / "figures"
print("SRC_DIR:", SRC_DIR)
print("ANALYSIS_DIR:", ANALYSIS_DIR)


def show(fig, name):
    # Hinh chi duoc luu neu ten thuoc publication.PUBLISHED_FIGURES (coverage matrix tham chieu dung cac ten nay).
    wave_a.save_figure(fig, FIGURES_DIR, name)
    plt.show()

## 7.1 Thu thap toan bo du lieu (1 pham vi connection duy nhat) + preflight/snapshot identity

`wave_a.collect_wave_a_data()` mo DUNG 1 `db.connect()`, chay het cac truy van (qua `queries.run_metric`, ket qua trong `data["m"]`),
roi dong connection TRUOC khi doc file (ownership/cohort manifest, holiday CSV). Assert non-terminal nam TRONG ham nay (fail nhanh).

In [ ]:
collect_overrides = {
    "ownership_manifest_path": EDA_OWNERSHIP_MANIFEST_PATH, "cohort_history_path": EDA_COHORT_HISTORY_PATH,
    "cohort_history_base_dir": EDA_COHORT_HISTORY_BASE_DIR, "vn_holidays_csv": EDA_VN_HOLIDAYS_CSV,
}
collect_overrides = {k: v for k, v in collect_overrides.items() if v is not None}
data = wave_a.collect_wave_a_data(pointer_path=EDA_POINTER_PATH, **collect_overrides)
snapshot = data["snapshot"]
print("database:", snapshot.database, "| batch_id:", snapshot.batch_id)
print("canonicalization:", snapshot.canonicalization_version, snapshot.canonicalization_git_commit)
print("metric da chay:", len(data["m"]), "/", len(queries.CATALOG), "| collect seconds:", data["pipeline_collect_seconds"])
display(data["core_counts"])

In [ ]:
manifest_overrides = {
    "warehouse_validation_report_path": EDA_WAREHOUSE_VALIDATION_REPORT_PATH,
    "ownership_manifest_path": EDA_OWNERSHIP_MANIFEST_PATH, "cohort_history_path": EDA_COHORT_HISTORY_PATH,
    "cohort_history_base_dir": EDA_COHORT_HISTORY_BASE_DIR, "source_manifest_path": EDA_SOURCE_MANIFEST_PATH,
}
manifest_overrides = {k: v for k, v in manifest_overrides.items() if v is not None}
input_manifest = wave_a.build_input_manifest(
    data, notebook_source_path=SRC_DIR.parent / "notebooks" / "01_warehouse_full_history_eda.ipynb",
    **manifest_overrides,
)
artifacts.atomic_write_json(ANALYSIS_DIR / "input_manifest.json", input_manifest)
print("input_manifest.json da ghi. code_provenance.is_dirty =", input_manifest["code_provenance"]["is_dirty"])
print("query_catalog_version:", input_manifest["query_catalog_version"], "| so metric:", len(input_manifest["query_catalog_metric_ids"]))
print("protocol_complete_through_date_by_source:", input_manifest["protocol_complete_through_date_by_source"])
print("coverage_matrix:", input_manifest["coverage_matrix"])

## Tinh toan BO bang (1 nguon duy nhat - GPT review 12 eda M4)

`wave_a.compute_wave_a_tables(data, input_manifest)` tinh HET moi bang vao MOT dict duy nhat (tap khoa == `publication.PUBLISHED_TABLES`).
Cac cell ben duoi chi `display(tables[...])` va ve hinh tu bang da aggregate - KHONG tu tinh lai hoac tu ghi CSV rieng.
`write_wave_a_tables()` (cuoi notebook) la noi DUY NHAT ghi CSV.

In [ ]:
tables = wave_a.compute_wave_a_tables(data, input_manifest)
print("so bang da tinh:", len(tables), "| so hinh se ve:", len(publication.PUBLISHED_FIGURES))
display(tables["preflight_reconciliation"])
display(tables["preflight_import_sources"])
display(tables["observation_date_ranges_main"])

## 7.2 Source, ownership va protocol coverage (+ collision audit RAW)

Moi ty le co numerator, denominator va dinh nghia denominator trong `TABLE_METADATA.csv`. Collision audit la RAW thuan tuy, KHONG doi MAIN;
gia khac nhau khi 2 nguon crawl o thoi diem khac nhau KHONG tu dong la loi parser.

In [ ]:
display(tables["ownership_by_source_status_reason"])
display(tables["raw_vs_main_by_source"])
show(plots.plot_run_item_by_source_crawl_date(tables["run_item_observation_by_source_crawl_date"]), "run_item_by_source_crawl_date")
show(plots.plot_raw_vs_main_by_source(tables["raw_vs_main_by_source"]), "raw_vs_main_by_source")
display(tables["protocol_continuity_summary"])
show(plots.plot_owner_outcome_rate_by_source_date(tables["protocol_outcome_rates_by_source_date"]), "owner_outcome_rate_by_source_date")

In [ ]:
for name in ("collision_item_summary", "collision_item_status_concordance", "collision_item_time_diff_stratification",
             "collision_option_summary", "collision_option_coverage_summary", "collision_option_time_diff_stratification"):
    print("==", name)
    display(tables[name])

## 7.3 Crawl operations va capacity

Khong dung thoi luong run de suy chat luong gia - day la quality/capacity metric rieng (muc 7.3). Ngay bat thuong chi la FLAG.

In [ ]:
run_duration = tables["run_duration_and_throughput"]
display(run_duration.groupby("source_code")[["duration_minutes", "items_per_hour", "observations_per_hour", "n_checkin_slots"]].describe().T)
print("so run keo qua ngay crawl ke tiep:", int(run_duration["crosses_next_crawl_day"].sum()), "/", len(run_duration))
show(plots.plot_run_duration_by_source(run_duration), "run_duration_by_source")
show(plots.plot_finish_hour_distribution(tables["finish_hour_distribution"]), "finish_hour_distribution")
flags = tables["run_day_operational_flags"]
display(flags[flags["is_any_anomalous"]])
display(tables["run_day_error_code_counts_raw"].head(20))

## 7.4 Hotel va check-in coverage

"Active hotel" = co it nhat 1 item DA DUOC OWN (owner_success/owner_failure) trong ngay - khong dung `hotels.booking_status` hien tai.

In [ ]:
active = tables["active_hotel_by_crawl_date_source"]
display(active.groupby("source_code")["n_active_hotels"].describe())
display(tables["active_hotel_by_crawl_date_source_city"].head(15))
display(tables["checkin_dates_tracked_by_crawl_date_source"].head(15))
display(tables["cohort_attrition_by_version"])
show(plots.plot_active_hotel_by_date(active), "active_hotel_by_date")
show(plots.plot_crawl_date_lead_time_heatmap(tables["crawl_date_lead_time_bucket_heatmap"]), "crawl_date_lead_time_heatmap")
show(plots.plot_checkin_coverage_weekday_month(tables["item_checkin_weekday_distribution_main"], tables["item_checkin_month_distribution_main"]),
     "checkin_coverage_weekday_month")

## 7.5 Protocol continuity va room/rate turnover (3 metric TACH RIENG)

1. **Protocol continuity**: lich EXPECTED tu ownership + cohort history (cutoff tren = `protocol_complete_through_date`), LEFT JOIN item that.
   `missing_source_run` (ca ngay khong co run) khac `missing_item_in_existing_run` (co run nhung thieu item).
2. **Canonical-series turnover**: chi mo ta ngay observed; ngay vang la `not_observed/turnover_unknown`, khong suy missing.
3. **Parser completeness**: xem missingness (7.9).

In [ ]:
display(tables["protocol_continuity_summary"])
display(tables["protocol_continuity_exceptions"].head(20))
display(tables["protocol_continuity_unattributed_errors"])
display(tables["canonical_series_turnover_by_observed_days"])
display(tables["canonical_series_max_gap_distribution"])
display(tables["canonical_series_median_gap_distribution"])
display(tables["canonical_series_turnover_by_series_main"].head(10))
display(tables["canonical_series_turnover_sample_main"].head(10))
show(plots.plot_series_turnover(tables["canonical_series_turnover_by_observed_days"], tables["canonical_series_max_gap_distribution"], tables["canonical_series_median_gap_distribution"]),
     "series_turnover_distributions")

## 7.6 Lead time va calendar coverage

Calendar chinh dung `holiday_date = checkin_date`. Holiday CSV da validate + aggregate TRUOC join (`holidays.py`), 1 dong DUY NHAT / `(checkin_date, city)`.
Join calendar vao observation xay ra TRONG SQL tren bang dem NHO - khong bao gio trong Python o grain observation.

In [ ]:
display(tables["item_lead_time_bucket_distribution_main"])
display(tables["item_lead_time_bucket_distribution_by_city_source_main"])
display(tables["item_calendar_coverage_main"])
display(tables["observation_counts_by_calendar_flags_main"])  # appendix: option-weighted
display(tables["holiday_calendar_flags_by_checkin_date_city"].head(10))
display(tables["vn_holidays_events_audit"].head(10))
show(plots.plot_lead_time_bucket_distribution(tables["item_lead_time_bucket_distribution_main"]), "lead_time_bucket_distribution")

## 7.7 Price distribution

Grain OBSERVATION (1 room option = 1 dong), tach RAW va MAIN. Histogram ve tu BIN COUNT SQL; box plot ve tu QUANTILE SQL (`Axes.bxp`) - khong keo observation
vao Python. Kem bang sensitivity o grain `(hotel_id, checkin_date, vn_observation_date)`.

In [ ]:
display(tables["price_distribution_overall_main"])
display(tables["price_distribution_overall_raw"])
display(tables["price_distribution_by_city_main"])
display(tables["price_distribution_by_lead_time_bucket_main"])
display(tables["price_distribution_by_weekday_main"])
display(tables["price_distribution_by_calendar_flags_main"])
display(tables["price_sensitivity_summary_main"])
display(tables["price_hotel_dispersion_main"].head(10))
display(tables["price_outlier_sample_main"].head(10))

In [ ]:
show(plots.plot_price_histograms(tables["price_histogram_linear_main"], tables["price_histogram_log10_main"]), "price_histogram_main")
show(plots.plot_price_box_by_city(tables["price_box_stats_by_city_main"]), "price_box_by_city")
show(plots.plot_price_by_lead_time_bucket(tables["price_distribution_by_lead_time_bucket_main"]), "price_by_lead_time_bucket")

## 7.8 Availability state (item grain - GPT review 12 M2)

Ty le status o grain ITEM (moi item dung 1 lan), khong dung so observation lam denominator. Moi status co mat ke ca = 0.

In [ ]:
present = metrics.status_present_report(tables["item_availability_overall"])
print("status co mat (0 nghia la KHONG bi bo qua):", present)
display(tables["item_availability_overall"])
display(tables["item_availability_by_city"])
display(tables["item_availability_by_checkin_month"])
display(tables["item_availability_by_lead_time_bucket"])
display(tables["item_availability_by_hotel"].sort_values("n_not_bookable", ascending=False).head(10))
display(tables["item_availability_by_crawl_date_hotel"].sort_values("n_not_bookable", ascending=False).head(10))
show(plots.plot_item_availability_stacked(tables["item_availability_by_city"], group_col="city",
                                          title="Ty le status item theo thanh pho (MAIN)"), "item_availability_by_city")
show(plots.plot_item_availability_stacked(tables["item_availability_by_lead_time_bucket"], group_col="lead_time_bucket",
                                          title="Ty le status item theo lead-time bucket (MAIN)"), "item_availability_by_lead_time")

## 7.9 Missingness va parser completeness

Structural missing (sentinel sold-out khong co room payload) da TACH khoi unexpected missing tren available observation
(`missing_kind` trong bang theo item status + sold-out).

In [ ]:
missingness = tables["missingness_available_observations"]
display(missingness.pivot_table(index=["field_group", "field"], columns="source_code", values="null_rate"))
display(tables["missingness_by_selector_version"].query("n_null > 0").head(20))
display(tables["missingness_by_item_status_sold_out"].query("n_null > 0").head(20))
display(tables["artifact_completeness_by_source_crawl_date"])
show(plots.plot_missingness_heatmap(missingness), "missingness_heatmap_by_source")

## 7.10 Full-history reference audit (GPT review 12 M3)

Hai metric KHAC dinh nghia, khong gop: **exact approved-key** (MAIN chinh, RAW phu luc) va **series-has-approved-reference** (metric long, bucket legacy 0-3 de doi chieu
bang lich su CLAUDE.md muc 7.2). Full-history turnover KHONG phai causal train coverage.

In [ ]:
print("=== Exact approved-key coverage, MAIN (bang chinh) ===")
display(tables["reference_exact_key_coverage_main"])
print("=== Exact approved-key coverage, RAW (phu luc) ===")
display(tables["reference_exact_key_coverage_raw"])
print("=== Series-exists coverage, RAW, bucket chuan ===")
display(tables["reference_series_exists_coverage_raw"])
print("=== Series-exists coverage, RAW, bucket LEGACY (0-3 gop) - doi chieu bang lich su ===")
display(tables["reference_series_exists_coverage_raw_legacy_bucket"])
print("=== Item-level exact-reference availability theo lead-time ===")
display(tables["reference_item_level_availability_by_lead_time"])
display(tables["reference_status_evidence_summary"])
display(tables["reference_uniqueness_per_series"])
display(tables["reference_candidate_coverage_summary"])
display(tables["reference_approval_by_city_month"].head(15))
show(plots.plot_reference_coverage_by_lead_time(
    tables["reference_exact_key_coverage_main"], tables["reference_series_exists_coverage_raw_legacy_bucket"]),
    "reference_coverage_by_lead_time")

## 7.11 Data quality findings

`quality_findings.csv` (`wave_a.build_quality_findings`): moi check co severity/scope/grain/count/denominator/rate/sample_keys/likely_cause/recommended_action,
ke ca check count=0 van co 1 dong. Gom outlier robust within-hotel (chi flag) va collision/source divergence.

In [ ]:
quality_findings = tables["quality_findings"]
display(quality_findings[["check_id", "severity", "scope", "grain", "count", "denominator", "rate"]])

## 7.12 Readiness cho dataset/model (chi bao readiness - GPT review 12 M4)

`dataset_readiness_by_horizon`: dem DUNG cap ngay quan sat cach nhau CHINH XAC K ngay (khong dung `n_observed_days >= K`); `actual_causal_labels` = NULL cho toi Wave B.

In [ ]:
display(tables["dataset_readiness_by_horizon"])
display(tables["history_length_by_hotel_checkin_main"].groupby("history_days_bucket")[["n_series"]].sum())
display(tables["series_evidence_runs_share"])
show(plots.plot_readiness_by_horizon(tables["dataset_readiness_by_horizon"]), "readiness_by_horizon")
show(plots.plot_history_length(tables["history_length_by_hotel_checkin_main"]), "history_length_distribution")

## Kiem tra trang thai Wave B (GPT review 12 M7 - dung dung dataset_version PASS)

In [ ]:
wave_b = tables["wave_b_dataset_version_readiness"]
display(wave_b)
if wave_b.empty or not bool(wave_b["ready"].any()):
    print("Wave B: CHUA CO dataset_version nao vua status='pass' vua du 3 bang ml_*. Notebook 02 chi scaffold, khong publish ket qua.")
else:
    print("Wave B: da co it nhat 1 dataset_version san sang - can chay Notebook 02 rieng.")

## Ghi cac artifact con lai (bang/report/dictionary/coverage matrix/summary) - manifest cuoi cung do `run_wave_a.py` ghi

In [ ]:
wave_a.write_wave_a_tables(tables, ANALYSIS_DIR)
wave_a.write_eda_summary(data, ANALYSIS_DIR, tables)
wave_a.write_eda_report_and_dictionary(data, tables, ANALYSIS_DIR, input_manifest=input_manifest)

missing_figures = [n for n in publication.PUBLISHED_FIGURES if not (FIGURES_DIR / f"{n}.png").exists()]
assert not missing_figures, f"hinh publish chua duoc ve: {missing_figures}"
print("Da ghi", len(tables), "bang,", len(publication.PUBLISHED_FIGURES), "hinh, eda_summary.json, EDA_REPORT.md, DATA_DICTIONARY.md, EDA_COVERAGE_MATRIX.md/.csv, TABLE_METADATA.csv.")
print("(artifact_manifest.json se do run_wave_a.py ghi sau khi notebook chay xong)")
for p in sorted(ANALYSIS_DIR.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(ANALYSIS_DIR))